# Entropic OT & Sinkhorn — From the Math to Running Code

*Foundational notes & demos from a Computational Optimal Transport reading seminar (Summer 2026). I'm learning this material — these pages are my working notes, not an expert guide.*

**Companion notebook to the Entropy and Sinkhorn field notes.** Same story, now runnable.

The two equations everything rests on:

$$L_C^{\varepsilon}(a,b) \;=\; \min_{P\in U(a,b)} \langle C,P\rangle - \varepsilon H(P), \qquad\qquad P^{\varepsilon} = \operatorname{diag}(u)\,K\,\operatorname{diag}(v),\quad K=e^{-C/\varepsilon}.$$

🫏 *In words:* find the cheapest plan, with a discount $\varepsilon$ for spreading out. The answer is always the cost-kernel $K$ with a row-dial $u$ and a column-dial $v$ — so we just hunt for $u,v$ by rescaling rows to $a$, then columns to $b$, repeat.

**Shapes:** $a\in\mathbb{R}^n$ (sources), $b\in\mathbb{R}^m$ (targets), $P\in\mathbb{R}^{n\times m}$ ($n\cdot m$ entries), dual $u,v$ ($n+m$ dials).

### What this notebook covers
0. Setup &nbsp; 1. Build the pieces ($a,b,C,K$) &nbsp; 2. Sinkhorn from scratch &nbsp; 3. Sanity-check vs POT
4. **Six demonstrations:** &nbsp; (1) $\varepsilon$-sweep blurry→sharp &nbsp; (2) 1D bell-curve coupling &nbsp; (3) convergence animation (the wobble) &nbsp; (4) exact vs entropic + cost gap &nbsp; (5) 2D point-cloud transport &nbsp; (6) speed scaling
5. Takeaways


## Setup


In [ ]:
!pip install pot -q

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
import ot, time

%matplotlib inline
plt.rcParams.update({"figure.dpi":110, "font.size":11, "axes.grid":False})
COUP = "viridis"
print("POT version:", ot.__version__)

## Build the pieces

We place $n$ sources and $m$ targets on a line in $[0,1]$, use **squared-distance cost** $C_{ij}=(x_i-y_j)^2$, and form the **Gibbs kernel** $K=e^{-C/\varepsilon}$.

🫏 $C$ = "how far / how expensive." $K$ = the same thing flipped into *desirability*: cheap routes glow bright, dear routes go dark.


In [ ]:
def normalize(v):
    v = np.asarray(v, float); return v / v.sum()

def mixture(x, mus, sigmas, ws):
    g = sum(w*np.exp(-(x-mu)**2/(2*s*s)) for mu, s, w in zip(mus, sigmas, ws))
    return g / g.sum()

n = m = 16
x = np.linspace(0, 1, n)
y = np.linspace(0, 1, m)
C = (x[:, None] - y[None, :])**2                    # cost table
a = mixture(x, [0.28, 0.70], [0.07, 0.06], [0.6, 0.4])   # bimodal source
b = mixture(y, [0.42, 0.86], [0.09, 0.05], [0.5, 0.5])   # bimodal target
eps = 0.02
K = np.exp(-C / eps)
print("a sums to", round(a.sum(),6), "| b sums to", round(b.sum(),6), "| shapes", C.shape)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8.2, 3.4))
im0 = ax[0].imshow(C, origin="lower", cmap="magma_r"); ax[0].set_title("cost  C  (dark = cheap)")
im1 = ax[1].imshow(K, origin="lower", cmap=COUP);      ax[1].set_title(r"kernel  $K=e^{-C/\varepsilon}$  (bright = desirable)")
for a_ in ax: a_.set_xlabel("target j"); a_.set_ylabel("source i")
fig.colorbar(im0, ax=ax[0], fraction=.046); fig.colorbar(im1, ax=ax[1], fraction=.046)
plt.tight_layout(); plt.show()

## Sinkhorn from scratch

The whole algorithm, reading exactly like the notes: $K=e^{-C/\varepsilon}$, then repeat $u=a/(Kv)$ and $v=b/(K^\top u)$.

🫏 squeeze the rows so each carries its supply $a$; squeeze the columns so each carries its demand $b$; repeat until both hold.


In [ ]:
def sinkhorn(a, b, C, eps, n_iter=1000, log=False):
    K = np.exp(-C / eps)
    u = np.ones_like(a)
    v = np.ones_like(b)
    history = []
    for it in range(n_iter):
        u = a / (K @ v)                 # fix rows:  row sums -> a
        v = b / (K.T @ u)               # fix cols:  col sums -> b
        if log:
            P = u[:, None] * K * v[None, :]
            history.append((P.sum(1).copy(), P.sum(0).copy(), P.copy()))
    P = u[:, None] * K * v[None, :]     # P = diag(u) K diag(v)
    return (P, history) if log else P

In [ ]:
P = sinkhorn(a, b, C, eps)
print("row sum error |P1 - a|   =", np.abs(P.sum(1) - a).max())
print("col sum error |P^T 1 - b| =", np.abs(P.sum(0) - b).max())
print("transport cost <C,P>      =", (P * C).sum())

## Sanity check vs POT

Our from-scratch plan should match `ot.sinkhorn`, and both should be close to the **exact** `ot.emd` as $\varepsilon$ shrinks. (POT's `M` is our `C`; POT's `reg` is our `\varepsilon`.)


In [ ]:
P_ours  = sinkhorn(a, b, C, eps)
P_pot   = ot.sinkhorn(a, b, C, reg=eps)
P_exact = ot.emd(a, b, C)

print("max |ours - POT| :", np.abs(P_ours - P_pot).max())
print("cost ours / POT  :", round((P_ours*C).sum(),6), "/", round((P_pot*C).sum(),6))
print("cost exact (P*)  :", round((P_exact*C).sum(),6), "  (entropic cost is always >= this)")

## Demonstrations


### Demo 1 — $\varepsilon$-sweep: blurry → sharp (the entropy story)

Shrink $\varepsilon$ and watch $P^{\varepsilon}$ tighten from a smeared cloud toward the sharp exact plan — and the transport cost drop toward the exact cost.

In [ ]:
N = 64
xs = np.linspace(0, 1, N)
aS = mixture(xs, [0.22, 0.58], [0.05, 0.06], [0.6, 0.4])
bS = mixture(xs, [0.40, 0.82], [0.07, 0.05], [0.5, 0.5])
CS = (xs[:, None] - xs[None, :])**2
PexS = ot.emd(aS, bS, CS)

epss = [1.0, 0.1, 0.02, 0.005]
fig, ax = plt.subplots(1, len(epss)+1, figsize=(3*(len(epss)+1), 3))
for k, e in enumerate(epss):
    Pe = sinkhorn(aS, bS, CS, e)
    ax[k].imshow(Pe, origin="lower", cmap=COUP)
    ax[k].set_title(f"$\\varepsilon$={e}\ncost={(Pe*CS).sum():.3f}"); ax[k].set_xticks([]); ax[k].set_yticks([])
ax[-1].imshow(PexS, origin="lower", cmap=COUP)
ax[-1].set_title(f"exact $P^*$\ncost={(PexS*CS).sum():.3f}"); ax[-1].set_xticks([]); ax[-1].set_yticks([])
plt.suptitle("smaller $\\varepsilon$  $\\rightarrow$  sharper plan, lower cost", y=1.05); plt.tight_layout(); plt.show()

In [ ]:
es = np.geomspace(2.0, 0.003, 25)
costs = [ (sinkhorn(aS, bS, CS, e)*CS).sum() for e in es ]
plt.figure(figsize=(6,3.2))
plt.semilogx(es, costs, "-o", ms=4, label=r"entropic cost $\langle C,P^\varepsilon\rangle$")
plt.axhline((PexS*CS).sum(), ls="--", color="crimson", label="exact cost")
plt.gca().invert_xaxis(); plt.xlabel(r"$\varepsilon$ (shrinking $\rightarrow$)"); plt.ylabel("transport cost")
plt.legend(); plt.title(r"entropic cost $\geq$ exact, $\rightarrow$ exact as $\varepsilon\rightarrow 0$"); plt.tight_layout(); plt.show()

### Demo 2 — 1D bell curves: green source → red target (your drawing, live)

Source $a$ = a narrow green bell; target $b$ = a wider red bell (same mass). The plan $P$ fills the square; its margins are pinned to $a$ (bottom) and $b$ (left).

In [ ]:
N = 100; t = np.linspace(0, 1, N)
aG = mixture(t, [0.25, 0.55], [0.05, 0.05], [0.6, 0.4])
bG = mixture(t, [0.45, 0.80], [0.06, 0.05], [0.45, 0.55])
CG = (t[:, None] - t[None, :])**2
PG = sinkhorn(aG, bG, CG, eps=0.0015)

fig = plt.figure(figsize=(5.6, 5.6))
gs  = fig.add_gridspec(2, 2, width_ratios=[1,4], height_ratios=[4,1], wspace=0.04, hspace=0.04)
axM = fig.add_subplot(gs[0,1]); axL = fig.add_subplot(gs[0,0], sharey=axM); axB = fig.add_subplot(gs[1,1], sharex=axM)
axM.imshow(PG.T, origin="lower", extent=[0,1,0,1], cmap=COUP, aspect="auto")
axM.set_xticks([]); axM.set_yticks([]); axM.set_title("the coupling $P^\\varepsilon$")
axB.plot(t, aG, color="seagreen"); axB.fill_between(t, aG, color="seagreen", alpha=.3)
axB.set_xlabel("source a (green)"); axB.set_yticks([])
axL.plot(bG, t, color="crimson"); axL.fill_betweenx(t, bG, color="crimson", alpha=.3)
axL.invert_xaxis(); axL.set_ylabel("target b (red)"); axL.set_xticks([])
plt.show()

### Demo 3 — Convergence animation: the wobble

Watch the row sums (bars) chase $a$ while we alternate. After **fix rows** the row-sums hit $a$ exactly and the columns drift; after **fix cols** the columns hit $b$ and rows drift — and the drift shrinks every round until both lock. *That* is Sinkhorn converging.

In [ ]:
# the bounce: one squeeze per frame, exactly like the interactive stepper in the notes
nb = 6
xb = np.linspace(0, 1, nb)
Cb = (xb[:, None] - xb[None, :])**2
ab = normalize([0.05, 0.26, 0.10, 0.30, 0.17, 0.12])
bb = normalize([0.22, 0.08, 0.28, 0.12, 0.10, 0.20])
Kb = np.exp(-Cb / 0.04)

frames = [("start", Kb.copy())]
M = Kb.copy()
for _ in range(9):
    M = M * (ab / M.sum(1))[:, None]; frames.append(("row", M.copy()))   # fix rows
    M = M * (bb / M.sum(0))[None, :]; frames.append(("col", M.copy()))   # fix cols

fig, ax = plt.subplots(figsize=(6.4, 6.4))
def bounce(k):
    ax.clear()
    action, Mk = frames[k]
    rs, cs = Mk.sum(1), Mk.sum(0); mx = Mk.max()
    ax.imshow(Mk, origin="upper", cmap=COUP, extent=[0, nb, nb, 0], vmin=0, vmax=mx)
    for i in range(nb):
        for j in range(nb):
            ax.text(j+0.5, i+0.5, f"{Mk[i,j]:.2f}", ha="center", va="center",
                    fontsize=7, color=("white" if Mk[i,j] > mx*0.5 else "black"))
    for i in range(nb):
        ok = abs(rs[i]-ab[i]) < 2e-3
        ax.text(nb+0.12, i+0.5, (f"{rs[i]:.2f} \u2713" if ok else f"{rs[i]:.2f} \u2192{ab[i]:.2f}"),
                va="center", ha="left", fontsize=9, family="monospace", color=("seagreen" if ok else "0.55"))
    for j in range(nb):
        ok = abs(cs[j]-bb[j]) < 2e-3
        ax.text(j+0.5, nb+0.12, (f"{cs[j]:.2f}\n\u2713" if ok else f"{cs[j]:.2f}\n\u2192{bb[j]:.2f}"),
                va="top", ha="center", fontsize=8, family="monospace", color=("seagreen" if ok else "0.55"))
    ax.text(nb+0.12, -0.25, "row \u03a3", fontsize=9, family="monospace", color="0.4")
    ax.text(-0.1, nb+0.12, "col \u03a3", fontsize=9, family="monospace", color="0.4", va="top", ha="right")
    ax.set_xticks([]); ax.set_yticks([]); ax.set_xlim(-0.3, nb+1.7); ax.set_ylim(nb+1.3, -0.5)
    title = {"start":"start: this is K  (neither margin locked)",
             "row":"fix ROWS  \u2192  rows locked \u2713   (columns drift)",
             "col":"fix COLS  \u2192  columns locked \u2713   (rows drift)"}[action]
    err = max(np.abs(rs-ab).max(), np.abs(cs-bb).max())
    ax.set_title(f"{title}\nmax margin error = {err:.4f}", fontsize=11)
anim = animation.FuncAnimation(fig, bounce, frames=len(frames), interval=750)
plt.close(fig)
HTML(anim.to_jshtml())

In [ ]:
# the oscillation as one curve: margin error shrinking each full round
_, hist = sinkhorn(ab, bb, Cb, 0.04, n_iter=15, log=True)
errs = [max(np.abs(r-ab).max(), np.abs(c-bb).max()) for (r, c, _) in hist]
plt.figure(figsize=(6,3.2))
plt.semilogy(range(1, len(errs)+1), errs, "-o", ms=4)
plt.xlabel("full round"); plt.ylabel("max margin error (log)")
plt.title("the oscillation dies out geometrically"); plt.tight_layout(); plt.show()

### Demo 4 — Exact vs entropic, side by side + the cost gap

$P^*$ (exact) is corner-sharp; $P^{\varepsilon}$ is blurry. The blur costs a little: $\langle C,P^{\varepsilon}\rangle \ge \langle C,P^*\rangle$, always.

In [ ]:
Pe = sinkhorn(a, b, C, 0.01)
fig, ax = plt.subplots(1, 2, figsize=(8, 3.6))
ax[0].imshow(P_exact, origin="lower", cmap=COUP); ax[0].set_title(f"exact $P^*$  (cost {(P_exact*C).sum():.4f})")
ax[1].imshow(Pe, origin="lower", cmap=COUP);      ax[1].set_title(f"entropic $P^\\varepsilon$  (cost {(Pe*C).sum():.4f})")
for a_ in ax: a_.set_xlabel("target j"); a_.set_ylabel("source i")
gap = (Pe*C).sum() - (P_exact*C).sum()
plt.suptitle(f"price of blur = extra cost = {gap:.4f}  (always >= 0)", y=1.04); plt.tight_layout(); plt.show()

### Demo 5 — 2D point clouds: move one blob to another

Real OT setting: two clouds of 2D points (uniform weights), cost = squared Euclidean distance. Sinkhorn returns a soft coupling; we draw a line for each significant $P_{ij}$ (thicker = more mass).

In [ ]:
rng = np.random.default_rng(3)
th  = np.linspace(0, 2*np.pi, 70, endpoint=False)
src = np.column_stack([np.cos(th), np.sin(th)]) * 1.3 + rng.normal(0, 0.05, (70, 2))
c1  = rng.normal([3.4,  1.2], 0.28, (35, 2))
c2  = rng.normal([3.4, -1.2], 0.28, (35, 2))
tgt = np.vstack([c1, c2])
wa = np.ones(len(src))/len(src); wb = np.ones(len(tgt))/len(tgt)
C2 = ((src[:, None, :] - tgt[None, :, :])**2).sum(-1)
P2 = sinkhorn(wa, wb, C2, eps=0.06)

plt.figure(figsize=(7.2, 4.2))
thr = P2.max()*0.12
for i in range(len(src)):
    for j in range(len(tgt)):
        if P2[i, j] > thr:
            plt.plot([src[i,0], tgt[j,0]], [src[i,1], tgt[j,1]],
                     color="gray", lw=7*P2[i,j]/P2.max(), alpha=0.3, zorder=1)
plt.scatter(*src.T, c="seagreen", s=26, zorder=2, label="source (ring)")
plt.scatter(*tgt.T, c="crimson",  s=26, zorder=2, label="target (2 clusters)")
plt.legend(loc="upper left"); plt.title("entropic transport: a ring splits to two clusters")
plt.gca().set_aspect("equal"); plt.tight_layout(); plt.show()

### Demo 6 — Speed scaling: exact vs Sinkhorn

Exact OT (`ot.emd`) is a combinatorial LP that scales roughly $O(n^3)$; Sinkhorn is $O(nm)$ matrix–vector products per iteration.

*Honest note:* on a CPU at small/medium $n$ they're comparable (NumPy's BLAS vs POT's fast C++ simplex). Sinkhorn's real wins are **(a)** GPU parallelism (the whole multiply at once), **(b)** very large $n$, and **(c)** differentiability — none of which exact OT offers.

In [ ]:
sizes = [100, 200, 400, 700, 1000, 1400]
t_emd, t_sink = [], []
for s in sizes:
    xx = np.linspace(0, 1, s); CC = (xx[:,None]-xx[None,:])**2
    av = np.ones(s)/s; bv = np.ones(s)/s
    t0=time.time(); ot.emd(av, bv, CC);              t_emd.append(time.time()-t0)
    t0=time.time(); sinkhorn(av, bv, CC, 0.01, 10); t_sink.append(time.time()-t0)
plt.figure(figsize=(6,3.4))
plt.plot(sizes, t_emd,  "-o", label="exact  ot.emd")
plt.plot(sizes, t_sink, "-o", label="Sinkhorn (10 iter)")
plt.xlabel("n (problem size)"); plt.ylabel("runtime (s)")
plt.legend(); plt.title("exact OT bends up faster as n grows"); plt.tight_layout(); plt.show()

## Takeaways

- **Primal → dual.** We want the plan $P\in\mathbb{R}^{n\times m}$ ($n\cdot m$ numbers). We actually solve for the dials $u\in\mathbb{R}^n,\,v\in\mathbb{R}^m$ ($n+m$ numbers), via $P=\operatorname{diag}(u)\,K\,\operatorname{diag}(v)$. (When $n=m$: $n^2$ vs $2n$.)
- **The algorithm is two divisions.** $u=a/(Kv)$, $v=b/(K^\top u)$, repeat. Each step fixes one margin; the other drifts a shrinking amount — the **wobble** — until both lock (Sinkhorn's theorem; guaranteed because $K>0$).
- **The $\varepsilon$ trade.** Small $\varepsilon$ → sharp, near-exact, stiffer; large $\varepsilon$ → fast, smooth, blurrier. Entropic cost is **always** $\ge$ exact.
- **Why bother.** Speed at scale (parallel, GPU) and a smooth gradient (trainable loss) — the two payoffs from the entropy notes, cashed in here.

🫏 *Know the pickup fee at each source and the dropoff fee at each target, and the whole route table is determined.*


---

# Appendix — Field notes (handwritten → markdown)

What follows is a shorter companion transcription. The full Entropy → Sinkhorn derivation (uniqueness Hessian, Lagrangian stationarity, pseudocode, why it works) lives on the [01. Entropy & Sinkhorn](01_entropy_sinkhorn.md) notes page. Main reference: Peyré & Cuturi, *Computational Optimal Transport* ([arXiv:1803.00567](https://arxiv.org/abs/1803.00567)).


## A. Recap: Monge and Kantorovich

### Monge problem

Find a map $T: X \to Y$ that moves mass $\mu$ to a target $\nu$ as cheaply as possible.

- $\mu,\nu$ — probability measures  
- $X$ — source space · $Y$ — target space

$$\min_T\left\{\int_X c\!\left(x,T(x)\right)\,d\mu(x):\; T_{\#}\mu=\nu\right\}$$

Push-forward (conserving mass):

$$(T_{\#}\mu)(A) := \mu\!\left(T^{-1}(A)\right)=\nu(A).$$

Monge cannot *split* mass: each source point goes to one target.

### Kantorovich problem

Relax to a coupling $\gamma$ (a joint plan) on $X\times Y$:

$$\min_{\gamma\in\Pi(\mu,\nu)}\int_{X\times Y} c\,d\gamma.$$

- $c$ — cost function  
- $\gamma$ — transport plan (in discrete OT: a matrix $P$)

Valid plans:

$$\Pi(\mu,\nu)=\bigl\{\gamma\in\mathcal{P}(X\times Y):\;(\pi_x)_{\#}\gamma=\mu,\;(\pi_y)_{\#}\gamma=\nu\bigr\}.$$

$\pi_x,\pi_y$ are the projections onto $X$ and $Y$.


### Monge vs Kantorovich (tiny example)

Sources $m_1=10$, $m_2=15$ and targets $m_A=5$, $m_B=20$ (total mass $25$). One valid plan:

$$\gamma=\begin{bmatrix}5&5\\0&15\end{bmatrix}.$$

Mass *can* split (source 1 sends $5$ to A and $5$ to B). That splitting is the Kantorovich flexibility Monge does not allow.


### Duality (pickup / dropoff fees)

Idea from the notes: $\min(\text{KP})=\max(\text{DP})$.

$$\max\Bigl(\int\phi\,d\mu+\int\psi\,d\nu\Bigr)=\min\int c\,d\gamma,$$

restricted by $\phi+\psi\le c$.

In words: the cost of two 1D arrays $(\phi,\psi)$ combined cannot exceed the cost of the actual route (2D matrix).

Computationally $\phi,\psi$ are arrays of prices (dual potentials) on source and target bins.


### When a Monge map exists

If $c(x,y)=h(x-y)$ with $h$ strictly convex (e.g. $|x-y|^2$), and $\phi+\psi=c$ at optimum, differentiating in $x$ gives

$$\nabla\phi(x)=\nabla h(x-y)\quad\Rightarrow\quad y=x-(\nabla h)^{-1}(\nabla\phi(x))=:T(x).$$

So under those conditions the optimal plan is induced by a deterministic map $T$.

*(Chapter 3 of the computational book, rewritten in discrete language: measures → vectors, plan $\gamma$ → matrix $P$.)*


## B. Discrete OT as a linear program

### The primal

Cheapest way to shift histogram $a$ to histogram $b$:

$$L_C(a,b)=\min_{P\in U(a,b)}\sum_{ij}C_{ij}P_{ij}=\min_{P\in U(a,b)}\langle C,P\rangle.$$

- $L_C$ — result of the minimization  
- $P$ — transport plan  
- $C$ — cost table  
- $U(a,b)$ — feasible set: non-negative matrices with **rows summing to $a$**, **columns summing to $b$**

A linear objective over linear constraints ⇒ a linear program. In code one often flattens $P$ to a vector $p$ and writes the margins as $Ap=\begin{bmatrix}a\\b\end{bmatrix}$.


### The dual ($c$-transforms)

Every LP has a twin. Dual potentials $h$ (two arrays stacked — source fees $f$ and target fees $g$):

$$L_C(a,b)=\max_{h:\,A^\top h\le c}\begin{bmatrix}a\\b\end{bmatrix}^\top h.$$

Constraint on every route $(i,j)$:

$$f_i+g_j\le C_{ij}.$$

**$c$-transform.** Given source prices $f$, the tightest feasible target prices are

$$(f^c)_j=\min_i(C_{ij}-f_i),$$

so

$$L_C(a,b)=\max_f\langle f,a\rangle+\langle f^c,b\rangle.$$

Tiny example from the notes: $C_{1A}=2$, $C_{2A}=7$, $f_1=1$, $f_2=3$ ⇒ highest feasible $g=\min(2-1,\,7-3)=1$.

**Complementary slackness.** At optimum, mass only flows on routes where the two dual prices add up *exactly* to the cost: $f_i+g_j=C_{ij}$.


## C. Entropy in Optimal Transport

### Exact OT is sharp — and combinatorial

Fundamental fact of LP: the cheapest plan is sharp & sparse — the optimum of a linear objective over a polytope sits at a **corner**.

**The problem (from the notes):** Exact OT is accurate but slow (combinatorial search). Idea: *blur* the plan $P$ so the solution is smooth instead of a hard corner.


### Entropy = a spread score

$$H(P)=-\sum_{ij}P_{ij}(\log P_{ij}-1).$$

$H$ is a scalar that measures blurriness / how spread out the plan is. Low $H$ ≈ sharp (few routes); high $H$ ≈ spread.

Example from the notes (masses $10,15$ / $5,20$ → normalized $a=[0.4,0.6]$, $b=[0.2,0.8]$):

$$P_1=\begin{bmatrix}0.2&0.2\\0&0.6\end{bmatrix}\quad(H\approx 1.95),\qquad
P_2=\begin{bmatrix}0.1&0.3\\0.1&0.5\end{bmatrix}\quad(H\approx 2.17).$$

Same margins; the denser plan scores higher entropy.


### Entropic OT: add $-\varepsilon H$

$$L_C^{\varepsilon}(a,b)=\min_{P\in U(a,b)}\langle C,P\rangle-\varepsilon H(P).$$

$\langle C,P\rangle$ is the original OT cost; $\varepsilon$ is the knob that controls blur.

Geometry: at $\varepsilon=0$ the optimum is pinned to a corner of the feasible set. The $-\varepsilon H$ term turns the flat triangle into a **smooth bowl**; the interior point slides inward as $\varepsilon\uparrow$.

**Uniqueness (full sketch).** From $H(P)=-\sum_{ij}P_{ij}(\log P_{ij}-1)$:

$$\frac{\partial H}{\partial P_{ij}}=-\log P_{ij},\qquad
\frac{\partial^2 H}{\partial P_{ij}^2}=-\frac{1}{P_{ij}},\qquad
\operatorname{Hess} H=-\operatorname{diag}(1/P_{ij}).$$

Curvature is negative wherever $P_{ij}>0$, so $H$ is concave, $-\varepsilon H$ is strictly convex, and there is exactly one best plan $P^{\varepsilon}$. (See the notes page for the write-up.)


### The trade-off

$$\langle C,P^{\varepsilon}\rangle\;\ge\;\langle C,P^{\star}\rangle=L_C(a,b).$$

Blurred plan (Sinkhorn) always costs at least as much as the exact plan (combinatorial search) — never less.

| Exact OT ($\varepsilon\to 0$) | Entropic OT ($\varepsilon>0$) |
|---|---|
| Sharp / sparse | Blurry / spread |
| ~$O(n^3)$ LP | Sinkhorn iterations |
| Hard for NNs (non-diff) | Smooth, unique, usable as a loss |


## D. Sinkhorn's algorithm (from the notes)

Set up the Lagrangian with duals $f$ (row / supply $P\mathbf{1}=a$) and $g$ (col / demand $P^\top\mathbf{1}=b$):

$$\mathcal{L}(P,f,g)=\langle C,P\rangle-\varepsilon H(P)-\langle f,P\mathbf{1}-a\rangle-\langle g,P^\top\mathbf{1}-b\rangle.$$

Stationarity ($\partial\mathcal{L}/\partial P_{ij}=0$) gives

$$C_{ij}+\varepsilon\log P_{ij}-f_i-g_j=0
\quad\Longrightarrow\quad
P_{ij}=e^{(f_i+g_j-C_{ij})/\varepsilon}=e^{f_i/\varepsilon}\,e^{-C_{ij}/\varepsilon}\,e^{g_j/\varepsilon}.$$

With $K=e^{-C/\varepsilon}$ (desirability: cheap routes near $1$, dear near $0$), $u=e^{f/\varepsilon}$, $v=e^{g/\varepsilon}$:

$$P=\operatorname{diag}(u)\,K\,\operatorname{diag}(v),\qquad
u\odot(Kv)=a,\quad v\odot(K^\top u)=b.$$

Two equations, two unknowns — solve by alternating:

```text
K = exp(-C / ε)
v = ones
repeat:
    u ← a / (K v)      # fix rows  → row-sum = a
    v ← b / (Kᵀ u)     # fix cols  → col-sum = b
return P = diag(u) K diag(v)
```

**Why it works.** There exists a $(u,v)$ pair that fixes both margins because $K=e^{-C/\varepsilon}$ is strictly positive — **Sinkhorn's theorem**. $P$ is updated alternately (fix row → fix col → …) and **stops as they converge (like oscillation)**.

Mass on route $i\to j$: $P_{ij}=u_i\,K_{ij}\,v_j$.
